# File I/O & Exception Handling



In [1]:
#include <iostream>
#include <fstream>
#include <sstream>
#include <string>
#include <vector>
#include <stdexcept>
#include <filesystem>
#include <iomanip>
using namespace std;

## 1. Text File I/O

In [2]:
// Write to a text file
{
    ofstream outFile("/tmp/students.txt");
    if (!outFile) throw runtime_error("Cannot open file for writing");

    outFile << "Alice 20 5.81\n";
    outFile << "Bob 22 4.97\n";
    outFile << "Chris 20 4.25\n";
    outFile << "Diana 21 5.14\n";
    cout << "File written successfully." << endl;
}  // outFile closes automatically (RAII)

File written successfully.


In [3]:
// Read back line by line
struct Student { string name; int age; double grade; };
vector<Student> students;

{
    ifstream inFile("/tmp/students.txt");
    if (!inFile) throw runtime_error("Cannot open file for reading");

    string line;
    while (getline(inFile, line)) {
        istringstream iss(line);
        Student s;
        if (iss >> s.name >> s.age >> s.grade) {
            students.push_back(s);
        }
    }
}

cout << "Read " << students.size() << " students:\n";
for (const auto& s : students) {
    printf("  %-10s age=%d  Grade=%.2f\n", s.name.c_str(), s.age, s.grade);
}

Read 4 students:
  Alice      age=20  Grade=5.81
  Bob        age=22  Grade=4.97
  Chris      age=20  Grade=4.25
  Diana      age=21  Grade=5.14


## 2. String Streams

In [4]:
// ostringstream — build strings
ostringstream oss;
oss << "Name: Alice, Age: 20, Grade: " << fixed << setprecision(1) << 5.81;
string result = oss.str();
cout << result << endl;

// istringstream — parse strings
string str_num = "100 200 300 400 500";
istringstream iss2(str_num);
int total = 0, val;
while (iss2 >> val) total += val;
cout << "Sum of parsed values: " << total << endl;

Name: Alice, Age: 20, Grade: 5.8
Sum of parsed values: 1500


## 3. Exception Handling

In [5]:
// Basic try/catch
auto safeDivide = [](double a, double b) -> double {
    if (b == 0) throw invalid_argument("Division by zero");
    return a / b;
};

vector<pair<double,double>> tests = {{10, 2}, {5, 0}, {9, 3}, {7, 0}};
for (const auto& [a, b] : tests) {
    try {
        cout << a << " / " << b << " = " << safeDivide(a, b) << endl;
    } catch (const invalid_argument& e) {
        cout << "Error: " << e.what() << " (a=" << a << ", b=" << b << ")" << endl;
    }
}

10 / 2 = 5
5 / 0 = Error: Division by zero (a=5, b=0)
9 / 3 = 3
7 / 0 = Error: Division by zero (a=7, b=0)


In [6]:
// Custom exception class
class InsufficientFundsError : public runtime_error {
    double required, available;
public:
    InsufficientFundsError(double req, double avail)
        : runtime_error("Insufficient funds"), required(req), available(avail) {}

    double shortfall() const { return required - available; }
    const char* what() const noexcept override {
        static char buf[128];
        snprintf(buf, sizeof(buf),
                 "Insufficient funds: need %.2f, have %.2f (shortfall %.2f)",
                 required, available, required - available);
        return buf;
    }
};

try {
    double balance = 50.0, amount = 120.0;
    if (amount > balance) throw InsufficientFundsError(amount, balance);
} catch (const InsufficientFundsError& e) {
    cout << e.what() << endl;
    cout << "You need $" << e.shortfall() << " more." << endl;
}

Insufficient funds: need 120.00, have 50.00 (shortfall 70.00)
You need $70 more.


## 4. RAII and Exception Safety

RAII (Resource Acquisition Is Initialization) ensures resources are released even when exceptions are thrown. File streams use RAII — they close automatically when they go out of scope.

In [7]:
// Custom RAII wrapper
class FileGuard {
    FILE* fp;
public:
    FileGuard(const char* path, const char* mode) : fp(fopen(path, mode)) {
        if (!fp) throw runtime_error(string("Cannot open ") + path);
        cout << "File opened: " << path << endl;
    }
    ~FileGuard() {
        if (fp) { fclose(fp); cout << "File closed automatically." << endl; }
    }
    FILE* get() { return fp; }
    FileGuard(const FileGuard&) = delete; 
};

try {
    FileGuard fg("/tmp/raii_test.txt", "w");
    fprintf(fg.get(), "RAII ensures this file closes properly!\n");
    // Even if an exception is thrown here, ~FileGuard() will be called
} catch (const exception& e) {
    cout << "Exception: " << e.what() << endl;
}

File opened: /tmp/raii_test.txt
File closed automatically.
